In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from typing import List, Literal
from typing_extensions import TypedDict
from dotenv import load_dotenv

# LangChain 관련 임포트
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# LangGraph 관련 임포트
from langgraph.graph import StateGraph, START, END

# 환경설정
load_dotenv()

if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('key check!!!!!')


<span style="font-size:14px">

## <span style="color: Gold"> **LangGraph**
- node : 해야할 일
- edge (조건에 따라 분기시킬 수 있음) : 다음에 어떤일을 할지 연결 (node를 연결)
- conditioanl Edge : 상황에 따라 다음 작업이 달라짐
- graph : node, edge를 관리
<br><br>
- 핵심
    - 상태(state)를 가진다
    - 상태를 입력으로 받아 작동하는 노드들을 만든다
    - 노드와 노드를 연결한다
    - 조건에 따라 다른 노드로 연결되는 엣지를 만든다
    - 그래프 전체를 컴파일해서 실행 가능한 프로그램으로 만든다
<br><br>
- state
    - “현재까지 어떤 정보가 축적되어 있는지 저장하는 공유 메모리”
    - LangGraph에서는 모든 노드가 이 State를 받아서 필요한 걸 쓰고, 새로운 데이터를 추가해서 다음 노드에 넘김.
    - 왜 필요함?
        - 내부 검색 결과를 다음 단계에서 써야 함
        - 웹 검색 결과도 다음 단계에서 써야 함
        - LLM이 생성한 답변도 마지막에 넣어야 함
    - 즉, 각 단계가 서로 데이터를 주고받는 그릇이 바로 상태.
<br><br>
- Node
    - 하나의 독립적인 작업(Task) 또는 함수
    - LangGraph에서는 모든 노드가 "입력 = 상태, 출력 = 상태 업데이트" 형태를 가짐.
    - 노드는 "순서도에서 하나의 박스"라고 보면 됨
<br><br>
- Conditional Edge
    - 노드의 실행 결과에 따라 다음에 갈 경로를 다르게 선택하는 길
    - 즉, if문을 그래프에 구현하는 것
    - 너의 코드에서는:
        - 내부 검색 결과가 있으면 → 바로 generate 노드
        - 내부 문서가 없으면 → web_search 노드로 이동
    - 즉, LLM 기반의 스마트한 if/else 흐름 제어가 가능해짐.
<br><br>
- Edge
    - 노드와 노드를 연결하는 선
    - 그래프에서 흐름을 보여주는 화살표
<br><br>
- compile() : 흐름도를 실행 가능한 프로그램으로 만드는 과정
<br><br>
- invoke : 해당 그래프를 실제로 실행해봐 라는 뜻

In [ ]:

# 조건부 엣지가 포함된 그래프
def conditional_graph():
    '''조건부 엣지가 포함된 LangGraph
    검색결과에 따라 다른 경로로 분기'''

    # 1. 상태 정의
    class ConditionalState(TypedDict) :
        question : str
        documents : List[Document]
        search_type : str
        answer : str

    # 내부문서 : load text or .. 기타 등등
    INTERNAL_DOCS = {
        '회사' : [Document(page_content="우리회사의 AI전략은 RAG시스템 구축입니다.")],
        '정책' : [Document(page_content="사내 데이터 보안 정책은 외부 공유 금지입니다.")]
    }

    # 노드함수들을 구현
    def internal_search_node(state:ConditionalState) -> dict :
        '''내부 문서 검색'''
        question = state['question']
        documents = []
        for keyword, docs in INTERNAL_DOCS.items():  # 딕셔너리 이므로, items() --> key,value 튜플 반환
            if keyword in question:
                documents.extend(docs)
        return {'documents' : documents, 'search_type':'internal'}
    

    def web_search_node(state:ConditionalState) -> dict :
        '''웹 검색(시뮬레이션)'''
        mock_result = Document(
            page_content=f"{state['question']}에 대한 웹검색 결과입니다.",
            metadata = {'source':'web'}
        )
        return {'documents':[mock_result], 'search_type':'web'}
    
    # 답변 생성
    def generate_node(state:ConditionalState) -> dict :
        '''답변생성'''
        llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
        context = '\n'.join([ doc.page_content for doc in state['documents']])
        prompt = ChatPromptTemplate.from_template(
            '''컨텍스트 : {context}\n\n질문 : {question}\n\n답변:'''
        )
        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({'context' : context, 'question': state['question']})
        return {'answer': f"[{state['search_type']}] {answer}"}
    

    # 조건 함수
    def decide_search_type(state:ConditionalState) -> Literal['generate', 'web_search']:
        '''검색결과에 따라 분기'''
        if state['documents']:
            return 'generate'
        else:
            return 'web_search'
        
    
    # 그래프 구축
    graph = StateGraph(ConditionalState)
    graph.add_node('internal_search', internal_search_node)
    graph.add_node('web_search', web_search_node)
    graph.add_node('generate', generate_node)

    graph.add_edge(START, 'internal_search')
    graph.add_conditional_edges(
        'internal_search',
        decide_search_type,
        {
            'generate' : 'generate',
            'web_search' : 'web_search'
        }
    )
    graph.add_edge('web_search', 'generate')
    graph.add_edge('generate', END)

    app = graph.compile()

    # 테스트 1 : 내부 문서에 있는 질문
    print('\n[테스트1] 내부문서가 있는 경우')
    result1 = app.invoke({
        'question' : '회사 AI 전략은?',
        'documents' : [],
        'search_type' : '',
        'answer' : ''
    })
    print(f"답변 : {result1['answer']}")


    # 테스트 2 : 내부문서가 없는 질문
    print('\n[테스트2] 내부문서가 없는 경우 -->  웹 검색')
    result2 = app.invoke({
        'question' : '오늘 날씨는?',
        'documents' : [],
        'search_type' : '',
        'answer' : ''
    })    
    print(f"답변 : {result2['answer']}")


# 조건부 분기 테스트
conditional_graph()



[테스트1] 내부문서가 있는 경우
답변 : [internal] 회사의 AI 전략은 RAG 시스템 구축입니다. RAG 시스템은 정보 검색과 생성 모델을 결합하여 효율적인 데이터 활용과 의사결정을 지원하는 시스템입니다. 이를 통해 우리는 더 나은 인사이트를 제공하고, 고객의 요구에 신속하게 대응할 수 있는 능력을 강화할 계획입니다.

[테스트2] 내부문서가 없는 경우 -->  웹 검색
답변 : [web] 오늘 날씨는 지역에 따라 다를 수 있습니다. 특정 지역의 날씨를 알고 싶으시면, 해당 지역을 말씀해 주시면 더 정확한 정보를 제공해 드릴 수 있습니다. 일반적으로 기온, 강수 확률, 바람 세기 등을 확인하는 것이 좋습니다.
